# Preparación y limpieza de datos

En esta sección, analizaremos los pasos necesarios para preparar y limpiar los datos para el análisis. La preparación de datos es un paso crucial en el proceso de análisis de datos, ya que garantiza que los datos sean precisos, consistentes y estén listos para el análisis.

In [118]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
df = pd.read_csv("../../data/raw/online_retail_II.csv" , encoding="ISO-8859-1")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


## Estandarizar columnas y tipos de datos

Vamos a estandarizar todas las columnas con los siguientes pasos:
- Eliminar espacios y poner en minúsculas todos los nombres de las columnas.
- Eliminar caracteres especiales de los nombres de las columnas.
- Reemplazar los espacios restantes con guiones bajos.

### Estandarizar nombres de columnas

In [119]:
df.columns = df.columns.str.strip().str.lower().str.replace({
    ' ': '_',
    '(': '',
    ')': '',
    'invoicedate': 'invoice_date',
    'stockcode': 'stock_code',
})
df.columns

Index(['invoice', 'stock_code', 'description', 'quantity', 'invoice_date',
       'price', 'customer_id', 'country'],
      dtype='str')

### Convertir tipos de datos

También convertiremos los tipos de datos de las columnas para asegurarnos de que sean apropiados para el análisis. Esto incluye la conversión de columnas de fecha al formato de fecha y hora y columnas numéricas al formato numérico apropiado.

In [120]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   invoice       541910 non-null  str    
 1   stock_code    541910 non-null  str    
 2   description   540456 non-null  str    
 3   quantity      541910 non-null  int64  
 4   invoice_date  541910 non-null  str    
 5   price         541910 non-null  float64
 6   customer_id   406830 non-null  float64
 7   country       541910 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [121]:
df = df.dropna(subset=['customer_id'])
df['customer_id'] = df['customer_id'].astype('int64')
df['invoice_date'] = pd.to_datetime(df['invoice_date'] , format='%m/%d/%y %H:%M')
df['country'] = df['country'].astype('category')

In [122]:
df.head()

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


## Manejo de valores faltantes

Manejaremos los valores faltantes en el conjunto de datos. Los valores faltantes pueden dar lugar a análisis inexactos y deben abordarse adecuadamente. Reemplazaremos los valores de descripción que faltan con "Unknown".

In [123]:
df['description'] = df['description'].fillna('Unknown')
df[df['description'] == 'Unknown']

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country


Descubriremos que no hay valores de NA en la columna de descripción. Esto se debe a que eliminamos las filas con valores de NA en la columna customer_id, que también tenían valores de NA en la columna de descripción.

## Eliminar transacciones no válidas o no relevantes

### Resumen

- Elimine filas duplicadas según subconjuntos de transacciones únicos.
- Eliminar registros de precios cero o negativos.
- Excluir filas de cantidades negativas que no sean de cancelación a menos que sean explícitamente válidas.

### ¿Qué vamos a hacer?

Bueno, en primer lugar, eliminaremos las filas duplicadas según los subconjuntos de transacciones únicos. Luego, eliminaremos los registros de precios cero o negativos. Finalmente, excluiremos las filas de cantidades negativas que no sean de cancelación a menos que sean explícitamente válidas.

### Eliminar filas duplicadas según subconjuntos de transacciones únicos

In [124]:
df.nunique()
print(f"Number of unique customers: {df['customer_id'].nunique()}")

Number of unique customers: 4372


In [125]:
df.drop_duplicates(inplace=True)

In [126]:
print(f"Number of unique customers after dropping duplicates: {df['customer_id'].nunique()}")

Number of unique customers after dropping duplicates: 4372


### Eliminar registros de precios cero o negativos

In [127]:
print(f"Number of rows with zero or negative price: {df[df['price'] <= 0].shape[0]}")

Number of rows with zero or negative price: 40


In [128]:
df = df[df['price'] > 0]

In [129]:
print(f"Number of rows with zero or negative price: {df[df['price'] <= 0].shape[0]}")

Number of rows with zero or negative price: 0


### Excluir filas de cantidad negativa que no sean de cancelación

In [130]:
print(f"Number of negative quantity rows (cancellations): {df[(df['quantity'] <= 0) & (~df['invoice'].str.startswith('C'))].shape[0]}")

Number of negative quantity rows (cancellations): 0


In [131]:
negative_quantity_rows = df[(df['quantity'] < 0) & (~df['invoice'].str.startswith('C'))]
print(f"Number of negative quantity rows (non-cancellations): {negative_quantity_rows.shape[0]}")

Number of negative quantity rows (non-cancellations): 0


## Conocimiento

¿Por qué no tenemos filas de cantidades negativas? Porque ya eliminamos las filas con valores NA en la columna customer_id, que también tenían valores NA en la columna descripción. Por lo tanto, no tenemos filas de cantidades negativas en el conjunto de datos.

## Crear banderas comerciales

### Resumen

- Agregue el indicador is_cancellation según el prefijo de factura "C".
- Agregue la bandera special_stock_code para ajustes, descuentos, envíos y muestras.
- Agregue un indicador valid_transaction para los registros retenidos en el análisis.

### ¿Qué vamos a hacer?

Crearemos indicadores comerciales para identificar tipos específicos de transacciones en el conjunto de datos. Esto incluye agregar un indicador is_cancellation basado en el prefijo de factura "C", un indicador special_stock_code para ajustes, descuentos, envíos y muestras, y un indicador valid_transaction para los registros retenidos en el análisis.

In [132]:
df["is_cancellation"] = df["invoice"].str.startswith('C')

In [133]:
unique_special_stock_codes = list(df[(df["stock_code"].str.len() <= 4)]["stock_code"].unique())
df["special_stock_code"] = df["stock_code"].isin(unique_special_stock_codes)

In [134]:
df['valid_transaction'] = df["valid_transaction"] = (
    (~df["is_cancellation"]) &
    (~df["special_stock_code"]) &
    (df["quantity"] > 0) &
    (df["price"] > 0) &
    (df["customer_id"].notna())
)
print(f"Number of valid transactions: {df['valid_transaction'].sum()}")

Number of valid transactions: 391162


In [135]:
df.info()

<class 'pandas.DataFrame'>
Index: 401565 entries, 0 to 541909
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   invoice             401565 non-null  str           
 1   stock_code          401565 non-null  str           
 2   description         401565 non-null  str           
 3   quantity            401565 non-null  int64         
 4   invoice_date        401565 non-null  datetime64[us]
 5   price               401565 non-null  float64       
 6   customer_id         401565 non-null  int64         
 7   country             401565 non-null  category      
 8   is_cancellation     401565 non-null  bool          
 9   special_stock_code  401565 non-null  bool          
 10  valid_transaction   401565 non-null  bool          
dtypes: bool(3), category(1), datetime64[us](1), float64(1), int64(2), str(3)
memory usage: 26.0 MB


In [136]:
df.head()

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,is_cancellation,special_stock_code,valid_transaction
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,False,False,True
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,True
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,False,False,True
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,True
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,True


## Normalizar y enriquecer el conjunto de datos.

### Resumen

- Manejar casos especiales como descuentos y envíos de forma consistente.
- Obtenga características útiles a nivel de transacción, como ventas totales, valor del pedido e indicadores de compra reciente.
- Cree nuevas columnas para análisis, como ventas totales, valor del pedido e indicadores de compras recientes.

### ¿Qué vamos a hacer?

Manejaremos casos especiales como descuentos y envíos de manera consistente, derivaremos características útiles a nivel de transacción, como ventas totales, valor de pedido e indicadores de antigüedad de compras, y crearemos nuevas columnas para análisis, como ventas totales, valor de pedido e indicadores de antigüedad de compras.

In [137]:
df["is_discounted"] = df["stock_code"] =='D'
df["total_price"] = np.where((df["quantity"] >0) & (df["price"] > 0), df["quantity"] * df["price"], 0)
df["total_price"] = df["total_price"].round(2)
df["last_purchase_date"] = df.groupby("customer_id")["invoice_date"].transform("max")
df["days_since_last_purchase"] = (df["last_purchase_date"].max() - df["last_purchase_date"]).dt.days

## Resumen

Creamos nuevas columnas para análisis, como ventas totales, fecha de última compra y días desde la última compra. Estas nuevas columnas nos ayudarán a comprender mejor el comportamiento del cliente y tomar decisiones más informadas.

### Nuevas columnas creadas:

- **is_discounted:** Un indicador que indica si la transacción es un descuento o no.
- **total_price:** El precio total de la transacción, calculado como cantidad * precio.
- **last_purchase_date:** La fecha de la última compra realizada por el cliente.
- **days_since_last_purchase:** El número de días desde la última compra realizada por el cliente.
- **special_stock_code:** Una bandera que indica si el código de stock es un código de stock especial o no.

## Construir un conjunto de datos a nivel de cliente

Construiremos un conjunto de datos a nivel de cliente agregando los datos a nivel de transacción. Esto nos permitirá analizar el comportamiento de los clientes y segmentarlos en función de sus patrones de compra.

### Nuevo conjunto de datos creado:

- **customer_id:** El identificador único de cada cliente.
- **recencia:** El número de días desde la última compra realizada por el cliente.
- **frecuencia:** El número de transacciones realizadas por el cliente.
- **monetario**: El monto total gastado por el cliente.
- **avg_order_value:** El valor promedio del pedido para el cliente, calculado como monetario/frecuencia.
- **avg_quantity:** La cantidad promedio comprada por el cliente, calculada como cantidad total/frecuencia.
- **discount_rate:** La tasa de descuento para el cliente, calculada como el total de transacciones con descuento / el total de transacciones.

In [138]:
customers = (
    df[
        (df["valid_transaction"] == True) |
        (df["is_discounted"] == True)
    ]
    .groupby("customer_id")
    .agg(
        recency=("days_since_last_purchase", "min"),
        frequency=("invoice", "nunique"),
        monetary=("total_price", "sum"),
        avg_order_value=("total_price", "mean"),
        avg_quantity=("quantity", "mean"),
        discount_rate=("is_discounted", "mean"),
    )
    .reset_index()
)

customers["recency"] = customers["recency"].astype(int)

print(f"Number of unique customers: {customers['customer_id'].nunique()}")
customers.describe().T

Number of unique customers: 4335


,count,mean,std,min,25%,50%,75%,max
customer_id,4335.0,15299.372549,1721.813812,12346.00000,13812.500000,15298.000000,16778.500000,18287.000000
recency,4335.0,89.565167,99.259235,0.00000,16.000000,49.000000,138.000000,373.000000
frequency,4335.0,4.262514,7.734539,1.00000,1.000000,2.000000,5.000000,206.000000
monetary,4335.0,2015.546168,8902.704309,3.75000,304.105000,661.520000,1631.475000,279138.020000
avg_order_value,4335.0,67.943067,1468.596011,2.13697,12.262510,17.562566,24.626138,77183.600000
avg_quantity,4335.0,45.645504,1204.927236,1.00000,6.043478,10.000000,14.750000,74215.000000
discount_rate,4335.0,0.000114,0.002911,0.00000,0.000000,0.000000,0.000000,0.142857


## Guarde el conjunto de datos preparado

Finalmente, guardaremos el conjunto de datos preparado en un archivo CSV para su posterior análisis. Esto nos permitirá acceder y analizar fácilmente los datos en el futuro.

- **`prepared_customers.csv`**: Contiene los datos del cliente preparados.
- **`prepared_transactions.csv`**: Contiene los datos de la transacción preparada.

Guardaremos los datos del cliente preparados en un archivo CSV llamado "prepared_customers.csv" y los datos de la transacción preparada en un archivo CSV llamado "prepared_transactions.csv". Esto nos permitirá acceder y analizar fácilmente los datos en el futuro.

In [139]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 4335 entries, 0 to 4334
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      4335 non-null   int64  
 1   recency          4335 non-null   int64  
 2   frequency        4335 non-null   int64  
 3   monetary         4335 non-null   float64
 4   avg_order_value  4335 non-null   float64
 5   avg_quantity     4335 non-null   float64
 6   discount_rate    4335 non-null   float64
dtypes: float64(4), int64(3)
memory usage: 237.2 KB


In [140]:
customers.to_csv("../../data/processed/prepared_customers.csv", index=False)
df[df["valid_transaction"] == True].to_csv("../../data/processed/prepared_transactions.csv", index=False)

## Conclusión de la fase de preparación de datos

La fase de preparación y limpieza de datos ha transformado con éxito el conjunto de datos sin procesar del comercio minorista en línea en un formato estructurado, limpio y listo para análisis. Siguiendo el plan planificado, los pasos clave ejecutados son:

* **Estandarización de columnas**: los nombres de las columnas se eliminaron, se pusieron en minúsculas y se formatearon con guiones bajos para mantener la coherencia (por ejemplo, `invoice`, `stock_code`, `customer_id`).

* **Conversión de tipo de datos**: las fechas se analizaron correctamente en objetos de fecha y hora (`invoice_date`), los ID de cliente se convirtieron a números enteros y los identificadores de país se optimizaron como tipos categóricos.

* **Gestión de valores faltantes**: se filtraron las filas que carecían de `customer_id`, lo que garantiza cálculos sólidos a nivel de cliente, mientras que los campos de descripción se normalizaron.

* **Eliminación de transacciones no válidas**: se eliminaron registros duplicados, se eliminaron anomalías de precios cero y negativos y se validaron las transacciones para garantizar análisis de comportamiento limpios.

* **Creación de indicadores comerciales**: se implementaron con éxito indicadores personalizados como `is_cancellation`, `special_stock_code`,`valid_transaction`y`is_discounted`para aislar el comportamiento de compra principal.

* **Ingeniería y agregación de funciones**: se derivaron métricas a nivel de transacción como`total_price`e indicadores de actualidad (`days_since_last_purchase`), lo que culminó en un conjunto de datos de resumen final a nivel de cliente que contiene métricas de RFM (recencia, frecuencia, monetaria) junto con valores promedio de pedidos y promedios de cantidad.

* **Exportación de datos preparados**: los resultados limpios se exportaron de forma segura como`prepared_customers.csv`y`prepared_transactions.csv`para impulsar el próximo análisis de datos exploratorios (EDA) y modelos de agrupación.

---

## Siguiente paso: Análisis exploratorio de datos (EDA)

La siguiente fase, Análisis de datos exploratorios (EDA), se centrará en descubrir conocimientos, patrones y distribuciones profundos de los conjuntos de datos recién preparados (`prepared_customers.csv` y `prepared_transactions.csv`). El camino propuesto es:

* **Análisis univariante de métricas de clientes**
* Visualice las distribuciones de valores recientes, de frecuencia y monetarios (RFM) en toda la base de clientes.
* Compruebe si hay asimetrías, valores atípicos y rangos de gasto típicos de los clientes.
* **Tendencias temporales y de transacciones**
* Analizar las tendencias del volumen de compras durante meses, días de la semana y horas del día.
*Evaluar estacionalidad o periodos pico en ventas.
* **Perspectivas geográficas y de productos**
* Examinar los países con mejor desempeño y su contribución al valor monetario general.
* Identificar los códigos de existencias y descripciones de productos más vendidos.
* **Análisis de correlación y relación**
* Explore las correlaciones entre la frecuencia de los pedidos de los clientes, el tamaño promedio de la cesta y el gasto total.
* Investigar patrones entre transacciones con descuento o códigos de acciones especiales.

Esta etapa de EDA establecerá una sólida base visual y estadística, lo que permitirá una configuración precisa para los próximos modelos de segmentación de clientes.